# 01 - Yogyakarta Data Audit and Preprocessing

This notebook prepares the daily DI Yogyakarta climate dataset for LSTM Autoencoder training.

Scope:

- Use `climate_data.csv` as the primary source.
- Filter to DI Yogyakarta station rows.
- Do not use flood labels.
- Engineer rainfall, wind, seasonal, and station features.
- Export a clean feature table for model training.


In [ ]:
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "Dataset" / "datasetIDNkaggle"
CLIMATE_CSV = DATA_DIR / "climate_data.csv"
STATION_CSV = DATA_DIR / "station_detail.csv"
PROVINCE_CSV = DATA_DIR / "province_detail.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

YOGYAKARTA_STATION_IDS = ["96851", "96855", "96859"]
BASELINE_TRAINING_STATION_IDS = ["96851", "96855"]

print(CLIMATE_CSV)
print(STATION_CSV)
print(PROVINCE_CSV)


In [ ]:
climate = pd.read_csv(CLIMATE_CSV, dtype={"station_id": "string"})
stations = pd.read_csv(STATION_CSV, dtype={"station_id": "string", "province_id": "string"})
provinces = pd.read_csv(PROVINCE_CSV, dtype={"province_id": "string"})

climate["date"] = pd.to_datetime(climate["date"], format="%d-%m-%Y", errors="coerce")
climate["station_id"] = climate["station_id"].astype(str)
stations["station_id"] = stations["station_id"].astype(str)

numeric_columns = ["Tn", "Tx", "Tavg", "RH_avg", "RR", "ss", "ff_x", "ddd_x", "ff_avg"]
for column in numeric_columns:
    climate[column] = pd.to_numeric(climate[column], errors="coerce")

station_metadata = stations.merge(provinces, on="province_id", how="left")

print("Climate shape:", climate.shape)
print("Station metadata shape:", station_metadata.shape)
display(climate.head())
display(station_metadata[station_metadata["station_id"].isin(YOGYAKARTA_STATION_IDS)])


In [ ]:
yogya = climate[climate["station_id"].isin(YOGYAKARTA_STATION_IDS)].copy()
yogya = yogya.merge(
    station_metadata[
        ["station_id", "station_name", "region_name", "latitude", "longitude", "province_name"]
    ],
    on="station_id",
    how="left",
)
yogya = yogya.sort_values(["station_id", "date"]).drop_duplicates(["station_id", "date"]).reset_index(drop=True)

coverage = (
    yogya.groupby(["station_id", "station_name", "region_name"])
    .agg(
        rows=("date", "size"),
        start_date=("date", "min"),
        end_date=("date", "max"),
        unique_days=("date", "nunique"),
    )
    .reset_index()
)
coverage["start_date"] = coverage["start_date"].dt.date
coverage["end_date"] = coverage["end_date"].dt.date

display(coverage)
print("Training stations with available rows:", sorted(yogya["station_id"].unique()))


In [ ]:
missing_summary = (
    yogya[["date", "station_id"] + numeric_columns]
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .rename("missing_percent")
    .reset_index()
    .rename(columns={"index": "column"})
)
display(missing_summary)

physical_ranges = {
    "Tn": (10.0, 40.0),
    "Tx": (15.0, 45.0),
    "Tavg": (10.0, 40.0),
    "RH_avg": (0.0, 100.0),
    "RR": (0.0, 500.0),
    "ss": (0.0, 15.0),
    "ff_x": (0.0, 60.0),
    "ff_avg": (0.0, 60.0),
    "ddd_x": (0.0, 360.0),
}

outlier_rows = []
for column, (lower, upper) in physical_ranges.items():
    mask = yogya[column].notna() & ((yogya[column] < lower) | (yogya[column] > upper))
    outlier_rows.append({"column": column, "outlier_count": int(mask.sum())})

outlier_summary = pd.DataFrame(outlier_rows)
display(outlier_summary)


In [ ]:
IMPORTANT_MISSING_COLUMNS = ["RR", "ff_x", "ff_avg", "RH_avg"]
IMPUTE_COLUMNS = ["Tn", "Tx", "Tavg", "RH_avg", "RR", "ss", "ff_x", "ddd_x", "ff_avg"]


def clean_physical_ranges(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = df.copy()
    for column, (lower, upper) in physical_ranges.items():
        mask = cleaned[column].notna() & ((cleaned[column] < lower) | (cleaned[column] > upper))
        cleaned.loc[mask, column] = np.nan
    return cleaned


def add_missing_flags(df: pd.DataFrame) -> pd.DataFrame:
    flagged = df.copy()
    for column in IMPORTANT_MISSING_COLUMNS:
        flagged[f"missing_{column}"] = flagged[column].isna().astype(int)
    return flagged


def impute_by_station(df: pd.DataFrame, max_gap_days: int = 3) -> pd.DataFrame:
    frames = []
    for station_id, group in df.sort_values(["station_id", "date"]).groupby("station_id", sort=False):
        g = group.copy().set_index("date")
        for column in IMPUTE_COLUMNS:
            g[column] = g[column].interpolate(method="time", limit=max_gap_days, limit_direction="both")
        frames.append(g.reset_index())
    return pd.concat(frames, ignore_index=True).sort_values(["station_id", "date"]).reset_index(drop=True)


In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    engineered = df.sort_values(["station_id", "date"]).copy()
    engineered["temp_range"] = engineered["Tx"] - engineered["Tn"]

    direction_rad = np.deg2rad(engineered["ddd_x"].fillna(0.0))
    engineered["ddd_x_sin"] = np.sin(direction_rad)
    engineered["ddd_x_cos"] = np.cos(direction_rad)

    day_angle = 2 * np.pi * engineered["date"].dt.dayofyear / 365.25
    engineered["day_of_year_sin"] = np.sin(day_angle)
    engineered["day_of_year_cos"] = np.cos(day_angle)

    engineered["station_96855"] = (engineered["station_id"].astype(str) == "96855").astype(int)

    frames = []
    for station_id, group in engineered.groupby("station_id", sort=False):
        g = group.sort_values("date").copy()
        g["rain_3d"] = g["RR"].rolling(window=3, min_periods=1).sum()
        g["rain_7d"] = g["RR"].rolling(window=7, min_periods=1).sum()
        g["rain_change_1d"] = g["RR"].diff().fillna(0.0)
        g["wind_change_1d"] = g["ff_x"].diff().fillna(0.0)
        frames.append(g)

    return pd.concat(frames, ignore_index=True).sort_values(["station_id", "date"]).reset_index(drop=True)


base_features = ["Tn", "Tx", "Tavg", "RH_avg", "RR", "ss", "ff_x", "ff_avg"]
model_features = [
    "Tn",
    "Tx",
    "Tavg",
    "RH_avg",
    "RR",
    "ss",
    "ff_x",
    "ff_avg",
    "temp_range",
    "rain_3d",
    "rain_7d",
    "rain_change_1d",
    "wind_change_1d",
    "ddd_x_sin",
    "ddd_x_cos",
    "day_of_year_sin",
    "day_of_year_cos",
    "station_96855",
    "missing_RR",
    "missing_ff_x",
    "missing_ff_avg",
    "missing_RH_avg",
]


In [ ]:
prepared = yogya[yogya["station_id"].isin(BASELINE_TRAINING_STATION_IDS)].copy()
prepared = clean_physical_ranges(prepared)
prepared = add_missing_flags(prepared)
prepared = impute_by_station(prepared, max_gap_days=3)
prepared = engineer_features(prepared)
prepared = prepared.dropna(subset=model_features).reset_index(drop=True)

output_columns = [
    "date",
    "station_id",
    "station_name",
    "region_name",
    "province_name",
] + model_features

processed_path = PROCESSED_DIR / "yogyakarta_weather_features.csv"
prepared[output_columns].to_csv(processed_path, index=False)

print("Prepared shape:", prepared.shape)
print("Saved:", processed_path)
display(prepared[output_columns].head())
display(prepared.groupby("station_id")["date"].agg(["min", "max", "count"]))


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

for station_id, group in prepared.groupby("station_id"):
    axes[0].plot(group["date"], group["RR"], label=station_id, alpha=0.8)
    axes[1].plot(group["date"], group["ff_x"], label=station_id, alpha=0.8)
    axes[2].plot(group["date"], group["Tavg"], label=station_id, alpha=0.8)

axes[0].set_title("Daily Rainfall RR - DI Yogyakarta")
axes[1].set_title("Maximum Wind Speed ff_x - DI Yogyakarta")
axes[2].set_title("Average Temperature Tavg - DI Yogyakarta")
for ax in axes:
    ax.legend()
    ax.set_xlabel("Date")

plt.tight_layout()
plt.show()
